# Monarch - Gate 2: 400-item corpus NAA scan (Kaggle GPU)

Runs the full text->fMRI cascade (gTTS -> WhisperX -> Llama-3.2-3B + Wav2Vec-BERT -> TRIBE v2)
over the four-category corpus and writes one NAA row per item. This is the only step in the
thesis that needs a GPU and the only one that cannot be repeated cheaply.

**Cost:** ~75 s/item x 400 = ~8.3 GPU-hours. A Kaggle session is 12 h, so it fits in one
run if nothing interrupts it. Quota is 30 GPU-h/week.

### Before you press Run All

1. **Settings -> Accelerator = GPU** (T4 or P100).
2. **Add-ons -> Secrets -> `HF_TOKEN`**, from an account that has accepted the
   Llama-3.2 licence. Without it the embedding stage 401s.
3. **Add data -> your corpus dataset**, containing `corpus.csv` (400 rows, built by
   `scripts/build_corpus.py`). Any dataset name works; the notebook globs for the file.
4. The branch below must be **pushed to GitHub**. Kaggle clones the repo; anything sitting
   uncommitted on the laptop does not exist here.
5. **Resuming a killed run:** upload the partial `corpus_naa.csv` as a dataset too. Cell 8
   seeds `/kaggle/working` from it and `batch_naa.py` skips every item already scanned.

### Rules this notebook does not break

- No `alpha_hat` is quoted. Calibration is a separate, free CPU step run after the scan.
- Undefined NAA rows are written empty and counted, never dropped or filled.
- The summary at the end reports counts only. Analysis happens in `analyze_corpus.py`.

In [ ]:
%%bash
# tribev2 declares torch>=2.5.1,<2.7 and Kaggle ships 2.10, so the image is installed against
# a version the model was never built for. Rather than branch per card, put the declared
# version in place first: cu121 wheels target sm_50 through sm_90, which covers both the T4
# (sm_75) and the P100 (sm_60) Kaggle hands out.
# Must run before any torch import, since a live kernel keeps whichever version it loaded.
set -e
nvidia-smi --query-gpu=name --format=csv,noheader | head -1 | sed 's/^/card: /'
pip install -q --index-url https://download.pytorch.org/whl/cu121   torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1
python -c "import torch; print('torch', torch.__version__); print('targets', torch.cuda.get_arch_list())"


In [ ]:
import torch

assert torch.cuda.is_available(), 'No GPU. Settings -> Accelerator -> GPU, then restart.'

# A card can be visible and still unusable: torch only runs kernels for the architectures its
# build targets, and a mismatch surfaces deep in the model rather than here.
major, minor = torch.cuda.get_device_capability(0)
arch = f'sm_{major}{minor}'
supported = torch.cuda.get_arch_list()
assert arch in supported, (
    f'{torch.cuda.get_device_name(0)} is {arch}, which torch {torch.__version__} '
    f'does not target. Built for: {supported}'
)
print(torch.__version__, torch.cuda.get_device_name(0), arch, 'OK')

In [ ]:
%%bash
# Code lives in /kaggle/temp, not /kaggle/working. Everything under working becomes kernel
# output, so cloning there put the whole repository plus tribev2 in every download.
# The branch matters: --carry-cols and the corpus-builder fixes live only there.
REPO_URL=https://github.com/brn-mwai/monarch.git
BRANCH=thesis/amendment-and-analysis-layer
mkdir -p /kaggle/temp
cd /kaggle/temp
rm -rf monarch tribev2
git clone -q --branch $BRANCH $REPO_URL monarch
git clone -q https://github.com/brn-mwai/tribev2.git
apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null
cd monarch && git log --oneline -1
# Fail here rather than after eight GPU-hours: an older checkout drops every carried column.
grep -q 'carry-cols' services/inference/scripts/batch_naa.py \
  && echo 'OK: batch_naa.py supports --carry-cols' \
  || { echo 'STOP: this checkout predates --carry-cols. Push the thesis branch first.'; exit 1; }

# tribev2 hardcodes compute_type="float16" (eventstransforms.py:108). ctranslate2 needs
# compute capability 7.0 for an efficient fp16 path, so on the P100 (sm_60) Kaggle also hands
# out, transcription dies with "Requested float16 compute type, but the target device or
# backend do not support efficient float16 computation". Pick by hardware instead of assuming
# the card, since which one you get is not under our control.
python - <<'PATCH'
from pathlib import Path

path = Path("/kaggle/temp/tribev2/tribev2/eventstransforms.py")
src = path.read_text()
old = '        compute_type = "float16"'
new = (
    '        # sm_70 and above have an efficient fp16 path in ctranslate2; sm_60 does not.
'
    '        compute_type = "float16" if torch.cuda.get_device_capability(0)[0] >= 7 else "float32"'
)
assert old in src, "compute_type line not found: tribev2 changed, re-check the patch"
path.write_text(src.replace(old, new, 1))
print("patched tribev2 compute_type for pre-Volta cards")
PATCH


In [ ]:
%%bash
set -e
# Pin the three torch packages so resolving tribev2's dependencies cannot pull a different
# build in behind us; everything else is free to resolve.
mkdir -p /kaggle/temp
printf 'torch==2.5.1
torchvision==0.20.1
torchaudio==2.5.1
' > /kaggle/temp/constraints.txt

# Oldest published version satisfying neuralset's exca>=0.5.20 floor, so the API it
# calls exists without jumping to a release it was never tested against.
pip install -q exca==0.5.21

# WITH dependencies this time. Installing tribev2 --no-deps left neuralset, neuraltrain,
# einops, moviepy, soundfile and julius missing, and the smoke test died on the first import.
pip install -q -c /kaggle/temp/constraints.txt /kaggle/temp/tribev2

# whisperx wants torch ~=2.8 and would drag the pinned build out; it only needs to run as a
# subprocess for word timings, so it goes in without its dependency graph.
# whisperx 3.8.x demands torch~=2.8 while tribev2 demands <2.7, so they cannot coexist and
# --no-deps was the old way round it. That left pyannote uninstalled, and whisperx imports it
# at module load, so the CLI died the moment tribev2 shelled out to it for word timings.
# 3.4.2 is the newest release whose floor (torch>=2.5.1, pyannote-audio>=3.3.2,
# ctranslate2<4.5.0) the pinned build satisfies, so it installs WITH its dependencies.
pip install -q -c /kaggle/temp/constraints.txt "whisperx==3.4.2"
pip install -q -c /kaggle/temp/constraints.txt nltk nibabel ujson mne torchmetrics

# whisperx pins ctranslate2<4.5.0, which links cuDNN 8, while torch 2.5.1+cu121 ships cuDNN 9.
# The mismatch does not surface at import: it appears when the transcriber loads a model on
# the GPU, as "Could not load library libcudnn_ops_infer.so.8". 4.5.0 is the release that
# moved to cuDNN 9, and faster-whisper accepts anything in >=4.0,<5, so the upper pin is
# simply stale for this environment. --no-deps keeps the resolution from touching torch.
pip install -q --no-deps "ctranslate2==4.5.0"

python -m spacy download en_core_web_lg -q
python -c "import neuralset, neuraltrain, tribev2; print('tribev2 stack imports OK')"
# tribev2 runs whisperx as a subprocess, so what matters is that the CLI imports, not that the
# package is present. This is the check that would have caught the pyannote failure at install
# time instead of part way into the first item's inference.
whisperx --help > /dev/null && echo "whisperx CLI imports OK"

In [ ]:
import importlib.metadata as md

# neuralset 0.0.2 declares exca>=0.5.20 and calls exca.steps.base.NoValue, which does not
# exist in 0.5.17. The previous notebook pinned 0.5.17 and rewrote neuralset's version guard
# to accept it, which silenced the check without supplying the API and failed at import.
# The versions actually resolved are printed here so a run states its own environment.
for pkg in ("exca", "neuralset", "neuraltrain", "tribev2", "torch", "numpy", "transformers"):
    try:
        print(f"{pkg:14s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:14s} NOT INSTALLED")

import exca.steps.base as base

assert hasattr(base, "NoValue"), (
    f"exca {md.version('exca')} has no steps.base.NoValue; neuralset needs >=0.5.20"
)
print("exca API check OK")


In [ ]:
import glob
import os

# Two sources, because the two launch paths differ. A run started from the editor carries the
# Kaggle secret; a version pushed through the API never does, since kernel-metadata.json has
# no field for secret attachments. Reading a private dataset first makes both paths work and
# keeps a pushed version from dying here before it can exercise anything else.
def _hf_token() -> str:
    for path in glob.glob('/kaggle/input/**/hf_token.txt', recursive=True):
        token = open(path, encoding='utf-8').read().strip()
        if token:
            print('HF token from dataset:', path)
            return token
    from kaggle_secrets import UserSecretsClient
    print('HF token from Kaggle secret')
    return UserSecretsClient().get_secret('HF_TOKEN')

os.environ['HF_TOKEN'] = _hf_token()
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['MONARCH_WHISPER_MODEL'] = 'small'
os.environ['MONARCH_WHISPER_DEVICE'] = 'cuda'
os.environ['MONARCH_WHISPER_COMPUTE'] = 'float16'
os.environ['MONARCH_WHISPERX_CMD'] = 'whisperx'
os.environ['MONARCH_TRIBE_DEVICE'] = 'cuda'
os.environ['PYTHONPATH'] = '/kaggle/temp/tribev2'
print('env set')

# ctranslate2 dlopens cuDNN at inference time rather than linking it, and torch ships those
# libraries inside site-packages/nvidia rather than on the system loader path. Without this,
# transcription dies with "Unable to load any of {libcudnn_cnn.so.9...}" only once a
# convolution actually runs. tribev2 spawns whisperx as a subprocess inheriting os.environ,
# so setting it here is what reaches the process that needs it.
import os
import nvidia.cublas
import nvidia.cudnn

_lib_dirs = [
    os.path.join(os.path.dirname(nvidia.cudnn.__file__), "lib"),
    os.path.join(os.path.dirname(nvidia.cublas.__file__), "lib"),
]
os.environ["LD_LIBRARY_PATH"] = ":".join(_lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")])

_cnn = [f for f in os.listdir(_lib_dirs[0]) if f.startswith("libcudnn_cnn")]
assert _cnn, f"no libcudnn_cnn in {_lib_dirs[0]}; ctranslate2 cannot run convolutions"
print("cuDNN libs on path:", _cnn[:3])


In [ ]:
%%bash
# The real test of the transcription path: run the whisperx CLI on a generated wav exactly as
# tribev2 does. Loading a model does not exercise cuDNN's convolution library, which is why
# the earlier in-process check passed while inference failed. tiny keeps this to about a
# minute; tribev2 itself uses large-v3.
set -e
cd /kaggle/temp
python - <<'EOF'
import math, struct, wave
with wave.open("/kaggle/temp/probe.wav", "w") as w:
    w.setnchannels(1); w.setsampwidth(2); w.setframerate(16000)
    w.writeframes(b"".join(struct.pack("<h", int(3000 * math.sin(i / 8))) for i in range(16000)))
print("probe.wav written")
EOF
# Same choice the patched tribev2 makes, so the probe tests what the run will do.
CT=$(python -c "import torch; print('float16' if torch.cuda.get_device_capability(0)[0] >= 7 else 'float32')")
echo "compute_type: $CT"
whisperx /kaggle/temp/probe.wav --model tiny --language en --device cuda --compute_type "$CT" --output_dir /kaggle/temp/probe_out --output_format json
echo "whisperx GPU transcription OK"


In [ ]:
%%bash
cd /kaggle/temp/monarch/services/inference
PYTHONPATH=/kaggle/temp/tribev2 python scripts/smoke_test.py

Expect `Model loaded on device: cuda:0`, `Predictions shape: (T, 20484)`, `Smoke test PASSED`.
If this fails, stop. Every later cell burns GPU hours on a broken cascade.

In [ ]:
%%bash
# Chapter 4 has to state the model's depth and how it treats subject identity. Record both
# from the loaded artifact, not from the run-folder name, and keep the JSON with the results.
cd /kaggle/temp/monarch/services/inference
PYTHONPATH=/kaggle/temp/tribev2 python scripts/verify_tribe_checkpoint.py \
  --load-model --out /kaggle/working/tribe_facts.json

In [ ]:
import csv
import glob
import shutil
from collections import Counter

matches = sorted(glob.glob('/kaggle/input/**/corpus.csv', recursive=True))
assert matches, 'corpus.csv not found. Add data -> your corpus dataset.'
corpus_src = matches[0]
shutil.copy(corpus_src, '/kaggle/working/corpus.csv')

with open('/kaggle/working/corpus.csv', newline='', encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))
counts = Counter(r['category'] for r in rows)
print(corpus_src, '->', len(rows), 'rows')
for category, n in sorted(counts.items()):
    print(f'  {category:24s} {n}')
assert len(rows) == 400, f'expected 400 rows, got {len(rows)}'
assert len(counts) == 4 and set(counts.values()) == {100}, f'category imbalance: {counts}'

# Seed the output from a previous partial run so batch_naa resumes instead of rescanning.
partials = sorted(glob.glob('/kaggle/input/**/corpus_naa.csv', recursive=True))
if partials:
    shutil.copy(partials[0], '/kaggle/working/corpus_naa.csv')
    with open('/kaggle/working/corpus_naa.csv', newline='', encoding='utf-8') as handle:
        done = sum(1 for _ in csv.DictReader(handle))
    print(f'resuming from {partials[0]}: {done} items already scanned')
else:
    print('no partial found; starting a fresh scan')

In [ ]:
%%bash
# --carry-cols: the scan is the one step that cannot be repeated cheaply, so every column a
# later analysis needs travels through it now. Adding one later means 8.3 GPU-hours again.
cd /kaggle/temp/monarch/services/inference
PYTHONPATH=/kaggle/temp/tribev2 python scripts/batch_naa.py \
  --csv /kaggle/working/corpus.csv \
  --text-col text \
  --outcome-col category \
  --carry-cols id,manipulative,credibility,partisan_intensity,source_dataset,word_count \
  --out /kaggle/working/corpus_naa.csv

In [ ]:
import csv
import statistics
from collections import Counter, defaultdict

with open('/kaggle/working/corpus_naa.csv', newline='', encoding='utf-8') as handle:
    scanned = list(csv.DictReader(handle))

print(f'rows scanned: {len(scanned)} / 400')

# The ratio form of NAA is undefined whenever either network mean sits below baseline. How
# often that happens is a finding in itself and goes in the methods, so it is counted here.
defined = [r for r in scanned if r.get('naa')]
print(f'NAA (ratio) defined: {len(defined)}  undefined: {len(scanned) - len(defined)}')

by_category = defaultdict(list)
for r in scanned:
    if r.get('naa_signed'):
        by_category[r['category']].append(float(r['naa_signed']))

print('\nsigned NAA (A_aff - A_del) per category, descriptive only:')
for category, values in sorted(by_category.items()):
    if not values:
        continue
    spread = statistics.stdev(values) if len(values) > 1 else float('nan')
    print(f'  {category:24s} n={len(values):3d}  median={statistics.median(values):+.4f}  sd={spread:.4f}')

print('\nclassification counts:', dict(Counter(r.get('classification', '') for r in scanned)))
print('\nNo effect is claimed here. Run scripts/analyze_corpus.py on this CSV for RQ I / RQ II.')

## When this finishes

Download from the right panel (Output): `corpus_naa.csv` and `tribe_facts.json`.

If the session died partway, `corpus_naa.csv` still holds every item scanned so far, flushed
per row. Publish it as a dataset, attach it to the next run, and cell 8 picks it up.

**Do not quote an `alpha_hat` from this run.** Calibration is a separate CPU step
(`scripts/calibrate_alpha.py`), and the two prior runs returned an interval straddling zero.
If it straddles again, that null is the result and gets reported with its power statement.